# Data Preprocessing

Dataset: *AI-Driven Resume Screening* (Kaggle, ~30,000 samples)  
Goal: clean, encode, split, and scale features for ML and DL modeling.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("ai_resume_screening.csv")
df.head()

## Exploratory Data Analysis

In [ ]:
print(f"Shape: {df.shape}")
df.info()

In [ ]:
df.describe()

In [ ]:
# Class distribution — imbalanced (~70% shortlisted)
df["shortlisted"].value_counts()

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

## Feature Selection

`github_activity` is excluded — it requires a separate data source (GitHub API) and cannot be extracted from a resume PDF.

In [ ]:
df = df.drop(columns=["github_activity"])

X = df.drop("shortlisted", axis=1)
y = df["shortlisted"].map({"Yes": 1, "No": 0})

print(f"Features ({X.shape[1]}): {X.columns.tolist()}")
print(f"Samples:  {X.shape[0]}")
print(f"Target distribution:\n{y.value_counts()}")

## Train / Validation / Test Split

Stratified 70 / 15 / 15 split to preserve class distribution.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train:      {X_train.shape[0]:,} samples (70%)")
print(f"Validation: {X_val.shape[0]:,} samples (15%)")
print(f"Test:       {X_test.shape[0]:,} samples (15%)")

## Encoding

`education_level` is ordinally encoded (High School < Bachelors < Masters < PhD) since it has a natural order.

In [ ]:
education_order = {"High School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3}

X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_test_enc  = X_test.copy()

for split in [X_train_enc, X_val_enc, X_test_enc]:
    split["education_level"] = split["education_level"].map(education_order)

X_train_enc.head()

## Class Weights

Computed from the training set to handle the 70/30 imbalance.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))

print(f"  Not Shortlisted (0): {cw[0]:.4f}")
print(f"  Shortlisted     (1): {cw[1]:.4f}")

## Feature Scaling

StandardScaler fitted on the training set only — applied to validation and test sets to prevent data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_val_scaled   = scaler.transform(X_val_enc)
X_test_scaled  = scaler.transform(X_test_enc)

print("Preprocessing complete.")
print(f"Train: {X_train_scaled.shape} | Val: {X_val_scaled.shape} | Test: {X_test_scaled.shape}")